# Week 50, Day 5: Topological sort

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w050d5.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
needs = {"shoes": ["socks", "trousers"], "socks": [], "trousers": ["pants"], "pants": [], "belt": ["trousers"]}

done, order = set(), []
while len(order) < len(needs):
    for item in sorted(needs):
        if item not in done and all(n in done for n in needs[item]):
            done.add(item)
            order.append(item)
            break
print(order)

In [ ]:
needs = {"shoes": ["socks", "trousers"], "socks": [], "trousers": ["pants"], "pants": [], "belt": ["trousers"]}

# turn "x needs y" into edges y -> x
before = {item: [] for item in needs}
for item, reqs in needs.items():
    for r in reqs:
        before[r].append(item)

def respects(order, before):
    position = {v: i for i, v in enumerate(order)}
    return all(position[u] < position[v] for u in before for v in before[u])

print(before)
print(respects(["pants", "socks", "trousers", "shoes", "belt"], before), respects(["socks", "shoes", "pants", "trousers", "belt"], before))

In [ ]:
from collections import deque

def kahn(before):
    indeg = {v: 0 for v in before}
    for u in before:
        for v in before[u]:
            indeg[v] += 1
    ready = deque(v for v in before if indeg[v] == 0)
    order = []
    while ready:
        u = ready.popleft()
        order.append(u)
        for v in before[u]:
            indeg[v] -= 1
            if indeg[v] == 0:
                ready.append(v)
    if len(order) < len(before):
        stuck = sorted(v for v in before if indeg[v] > 0)
        raise ValueError(f"a cycle among {stuck}")
    return order

before = {"socks": ["shoes"], "pants": ["trousers"], "trousers": ["shoes", "belt"], "shoes": [], "belt": []}
print(kahn(before))
try:
    kahn({"a": ["b"], "b": ["c"], "c": ["a"], "d": ["a"]})
except ValueError as e:
    print("ValueError:", e)

In [ ]:
def dfs_topo(before):
    WHITE, GREY, BLACK = 0, 1, 2
    colour = {v: WHITE for v in before}
    finished = []
    for s in before:
        if colour[s] != WHITE:
            continue
        colour[s] = GREY
        stack = [(s, iter(before[s]))]
        while stack:
            v, it = stack[-1]
            for w in it:
                if colour[w] == GREY:
                    raise ValueError("cycle")
                if colour[w] == WHITE:
                    colour[w] = GREY
                    stack.append((w, iter(before[w])))
                    break
            else:
                colour[v] = BLACK
                finished.append(v)
                stack.pop()
    return finished[::-1]

before = {"socks": ["shoes"], "pants": ["trousers"], "trousers": ["shoes", "belt"], "shoes": [], "belt": []}
print(dfs_topo(before))

In [ ]:
from graphlib import TopologicalSorter, CycleError

needs = {"shoes": {"socks", "trousers"}, "trousers": {"pants"}, "belt": {"trousers"}}
print(list(TopologicalSorter(needs).static_order()))

ts = TopologicalSorter(needs)
ts.prepare()
rounds = []
while ts.is_active():
    ready = sorted(ts.get_ready())
    rounds.append(ready)
    ts.done(*ready)
print(rounds)

try:
    list(TopologicalSorter({"a": {"b"}, "b": {"a"}}).static_order())
except CycleError as e:
    print("CycleError:", e.args[1])

In [ ]:
from graphlib import TopologicalSorter

tasks = {                      # task: (days, needs)
    "design":  (5, []),
    "order":   (2, ["design"]),
    "foundations": (4, ["design"]),
    "frame":   (6, ["foundations", "order"]),
    "wiring":  (3, ["frame"]),
    "plumbing": (2, ["frame"]),
    "finish":  (4, ["wiring", "plumbing"]),
}
order = list(TopologicalSorter({t: set(n) for t, (_, n) in tasks.items()}).static_order())
earliest, via = {}, {}
for t in order:
    days, needs = tasks[t]
    start = max((earliest[n] for n in needs), default=0)
    via[t] = max(needs, key=lambda n: earliest[n], default=None)
    earliest[t] = start + days
end = max(earliest, key=earliest.get)
path = []
while end:
    path.append(end)
    end = via[end]
print(earliest, " -> ".join(reversed(path)))

> **Key idea.** A topological order lists every vertex before everything it points to; it exists exactly when the graph is a DAG, and there are usually several. Kahn's algorithm counts in-degrees, repeatedly takes a vertex with none, and lowers its neighbours' counts: O(V + E), and fewer than V vertices out means a cycle; a heap instead of a queue picks among ready vertices by a rule. Reversing DFS finish order also gives a topological order. `graphlib.TopologicalSorter` takes "depends on" sets, gives `static_order()` and rounds of ready tasks, and raises `CycleError`. Processing a DAG in topological order computes levels, longest paths and a project's critical path in one pass.

### Exercise: Kahn's algorithm

Write `kahn(before)` for a graph given as "u before v" lists (every vertex is a key): a topological order, taking ready vertices **in the order they become ready** (start with the zero in-degree vertices in the dict's order), or raise `ValueError` if there's a cycle. Then `stuck(before)`: the sorted list of vertices that can never be done because of a cycle (those left with in-degree above 0).

In [ ]:
from collections import deque

def kahn(before):
    ___

def stuck(before):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef respects(order, g):\n    pos = {v: i for i, v in enumerate(order)}\n    return sorted(order, key=str) == sorted(g, key=str) and all(pos[u] < pos[v] for u in g for v in g[u])\ndef test_kahn():\n    \"orders\"\n    before = {\"socks\": [\"shoes\"], \"pants\": [\"trousers\"], \"trousers\": [\"shoes\", \"belt\"], \"shoes\": [], \"belt\": []}\n    assert kahn(before) == [\"socks\", \"pants\", \"trousers\", \"shoes\", \"belt\"]\n    cyc = {\"a\": [\"b\"], \"b\": [\"c\"], \"c\": [\"a\"], \"d\": [\"a\"], \"e\": []}\n    try:\n        kahn(cyc)\n        assert False\n    except ValueError:\n        pass\n    assert stuck(cyc) == [\"a\", \"b\", \"c\"] and stuck(before) == []\n    rng = random.Random(1)\n    for _ in range(100):\n        n = rng.randint(1, 9)\n        perm = rng.sample(range(n), n)\n        g = {v: [] for v in range(n)}\n        for i in range(n):\n            for j in range(i + 1, n):\n                if rng.random() < 0.3:\n                    g[perm[i]].append(perm[j])\n        assert respects(kahn(g), g)")

### Exercise: The alphabetically first order

Write `first_order(needs)`, where `needs` maps each item to the list of items it needs (every item is a key): the topological order that is **alphabetically smallest**, taking the smallest ready item each time with `heapq`. Raise `ValueError` on a cycle. Then `unique_order(needs)`: whether there's **only one** topological order (true exactly when, at every step, just one item is ready).

In [ ]:
import heapq

def first_order(needs):
    ___

def unique_order(needs):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import itertools, random\ndef all_orders(needs):\n    out = []\n    for p in itertools.permutations(needs):\n        pos = {v: i for i, v in enumerate(p)}\n        if all(pos[r] < pos[v] for v in needs for r in needs[v]):\n            out.append(list(p))\n    return out\ndef test_first():\n    \"smallest\"\n    needs = {\"shoes\": [\"socks\", \"trousers\"], \"socks\": [], \"trousers\": [\"pants\"], \"pants\": [], \"belt\": [\"trousers\"]}\n    assert first_order(needs) == [\"pants\", \"socks\", \"trousers\", \"belt\", \"shoes\"]\n    assert unique_order({\"c\": [\"b\"], \"b\": [\"a\"], \"a\": []}) and not unique_order(needs)\n    try:\n        first_order({\"a\": [\"b\"], \"b\": [\"a\"]})\n        assert False\n    except ValueError:\n        pass\n    rng = random.Random(2)\n    for _ in range(80):\n        names = list(\"abcdef\")[:rng.randint(1, 6)]\n        perm = rng.sample(names, len(names))\n        needs = {v: [] for v in names}\n        for i in range(len(perm)):\n            for j in range(i + 1, len(perm)):\n                if rng.random() < 0.3:\n                    needs[perm[j]].append(perm[i])\n        orders = all_orders(needs)\n        assert first_order(needs) == min(orders) and unique_order(needs) == (len(orders) == 1)")

### Exercise: From finish times

Write `dfs_topo(before)`: a topological order by reversing DFS finish order, with an explicit stack (the checks use a 20,000-vertex chain), raising `ValueError` with the cycle's vertices in the message if a grey vertex is met. Then `is_topological(order, before)`: whether `order` contains every vertex exactly once and respects every edge.

In [ ]:
def dfs_topo(before):
    ___

def is_topological(order, before):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_dfs():\n    \"dfs order\"\n    before = {\"socks\": [\"shoes\"], \"pants\": [\"trousers\"], \"trousers\": [\"shoes\", \"belt\"], \"shoes\": [], \"belt\": []}\n    o = dfs_topo(before)\n    assert is_topological(o, before) and not is_topological([\"shoes\"] + o[:-1], before) and not is_topological(o[:-1], before)\n    try:\n        dfs_topo({1: [2], 2: [3], 3: [1]})\n        assert False\n    except ValueError:\n        pass\n    rng = random.Random(3)\n    for _ in range(100):\n        n = rng.randint(1, 9)\n        perm = rng.sample(range(n), n)\n        g = {v: [] for v in range(n)}\n        for i in range(n):\n            for j in range(i + 1, n):\n                if rng.random() < 0.3:\n                    g[perm[i]].append(perm[j])\n        assert is_topological(dfs_topo(g), g)\n    chain = {i: [i + 1] for i in range(20_000)}\n    chain[20_000] = []\n    assert dfs_topo(chain)[:3] == [0, 1, 2]")

### Exercise: Planning semesters

Courses have prerequisites: `prereqs` maps each course to the list of courses that must be passed **before** it (every course is a key). Write `semesters(prereqs)`: the fewest semesters needed if you can take any number of courses at once, and `plan(prereqs)`: a list of semesters, each a sorted list of courses, taking every course as early as possible. Use `graphlib.TopologicalSorter`, and let its `CycleError` become a `ValueError("impossible: ...")`. Then `plan_with_limit(prereqs, k)`: at most k courses a semester, greedily taking the ready courses with the **longest chain of courses depending on them** first (ties alphabetical).

In [ ]:
from graphlib import TopologicalSorter, CycleError

def semesters(prereqs):
    ___

def plan(prereqs):
    ___

def plan_with_limit(prereqs, k):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def valid(p, prereqs, k=None):\n    when = {c: i for i, sem in enumerate(p) for c in sem}\n    return sorted(when) == sorted(prereqs) and all(when[r] < when[c] for c in prereqs for r in prereqs[c]) and (k is None or all(len(s) <= k for s in p))\ndef test_plan():\n    \"semesters\"\n    pre = {\"maths1\": [], \"prog1\": [], \"maths2\": [\"maths1\"], \"prog2\": [\"prog1\"], \"algo\": [\"prog2\", \"maths2\"], \"project\": [\"algo\"], \"art\": []}\n    assert semesters(pre) == 4 and plan(pre) == [[\"art\", \"maths1\", \"prog1\"], [\"maths2\", \"prog2\"], [\"algo\"], [\"project\"]]\n    p = plan_with_limit(pre, 2)\n    assert valid(p, pre, 2) and len(p) == 4 and p[0] == [\"maths1\", \"prog1\"]\n    assert valid(plan_with_limit(pre, 1), pre, 1) and len(plan_with_limit(pre, 1)) == 7\n    assert semesters({}) == 0 and plan({}) == []\n    try:\n        plan({\"a\": [\"b\"], \"b\": [\"a\"]})\n        assert False\n    except ValueError as e:\n        assert \"impossible\" in str(e)")

### Exercise: The critical path

`tasks` maps each task to `(days, needs)`. Write `schedule(tasks)`: a dict from task to `(earliest_start, earliest_finish)`, processing tasks in topological order; `project_length(tasks)`; and `critical_path(tasks)`: the chain of tasks that sets the project's length, from first to last, choosing, where several needed tasks tie for the latest finish, the alphabetically first. Then `slack(tasks)`: a dict from task to how many days it could be delayed without delaying the project (0 on the critical path), from a second, backwards pass computing latest finish times.

In [ ]:
from graphlib import TopologicalSorter

def schedule(tasks):
    ___

def project_length(tasks):
    ___

def critical_path(tasks):
    ___

def slack(tasks):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("TASKS = {\n    \"design\": (5, []),\n    \"order\": (2, [\"design\"]),\n    \"foundations\": (4, [\"design\"]),\n    \"frame\": (6, [\"foundations\", \"order\"]),\n    \"wiring\": (3, [\"frame\"]),\n    \"plumbing\": (2, [\"frame\"]),\n    \"finish\": (4, [\"wiring\", \"plumbing\"]),\n}\ndef test_cpm():\n    \"the critical path\"\n    s = schedule(TASKS)\n    assert s[\"frame\"] == (9, 15) and s[\"finish\"] == (18, 22) and s[\"order\"] == (5, 7)\n    assert project_length(TASKS) == 22 and critical_path(TASKS) == [\"design\", \"foundations\", \"frame\", \"wiring\", \"finish\"]\n    assert slack(TASKS) == {\"design\": 0, \"order\": 2, \"foundations\": 0, \"frame\": 0, \"wiring\": 0, \"plumbing\": 1, \"finish\": 0}\n    assert project_length({}) == 0 and critical_path({}) == []\n    two = {\"a\": (3, []), \"b\": (3, [])}\n    assert critical_path(two) == [\"a\"] and slack(two) == {\"a\": 0, \"b\": 0}")

### Bug hunt: Ready too soon

This Kahn's algorithm queues a vertex as soon as **any** edge into it is processed, instead of when its last requirement is done, so "shoes" comes out before "trousers", and twice: this prints `['socks', 'pants', 'shoes', 'trousers', 'shoes', 'belt']`. Queue a vertex only when its in-degree reaches 0, so it prints `['socks', 'pants', 'trousers', 'shoes', 'belt']`.

In [ ]:
from collections import deque

before = {"socks": ["shoes"], "pants": ["trousers"], "trousers": ["shoes", "belt"], "shoes": [], "belt": []}
indeg = {v: 0 for v in before}
for u in before:
    for v in before[u]:
        indeg[v] += 1
ready = deque(v for v in before if indeg[v] == 0)
order = []
while ready:
    u = ready.popleft()
    order.append(u)
    for v in before[u]:
        indeg[v] -= 1
        ready.append(v)
print(order)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"only when free\"\n    expect_output(\"['socks', 'pants', 'trousers', 'shoes', 'belt']\")")

### Bug hunt: A cycle that goes unnoticed

This Kahn's algorithm returns whatever it managed to order, so for a graph with a cycle it quietly returns a **partial** order, leaving out the stuck tasks: this prints `['d']`. Check that every vertex came out and raise `ValueError` otherwise, so it prints `cycle`.

In [ ]:
from collections import deque

def kahn(before):
    indeg = {v: 0 for v in before}
    for u in before:
        for v in before[u]:
            indeg[v] += 1
    ready = deque(v for v in before if indeg[v] == 0)
    order = []
    while ready:
        u = ready.popleft()
        order.append(u)
        for v in before[u]:
            indeg[v] -= 1
            if indeg[v] == 0:
                ready.append(v)
    return order

try:
    print(kahn({"a": ["b"], "b": ["a"], "d": ["a"]}))
except ValueError:
    print("cycle")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"check the count\"\n    expect_output(\"cycle\")")

### Bug hunt: Discovery isn't finishing

This DFS-based sort records each vertex when it's **discovered** and uses that order, which isn't a topological order: "c" ends up before "b", although b → c. This prints `['a', 'c', 'b']`. Record vertices when they **finish**, then reverse, so it prints `['a', 'b', 'c']`.

In [ ]:
before = {"a": ["c", "b"], "b": ["c"], "c": []}
seen, order = set(), []

def visit(v):
    seen.add(v)
    order.append(v)
    for w in before[v]:
        if w not in seen:
            visit(w)

for v in before:
    if v not in seen:
        visit(v)
print(order)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"finish order, reversed\"\n    expect_output(\"['a', 'b', 'c']\")")

### Puzzle: Kahn's algorithm

Arrange the lines into Kahn's algorithm. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
# Edges (u before v): a -> b, a -> c, b -> d, c -> d, d -> e.

p = ___   # the in-degree of d
q = ___   # the vertices with in-degree 0 at the start, as a list
r = ___   # the number of different topological orders
s = ___   # the fewest rounds if everything ready happens at once
t = ___   # True or False: adding the edge e -> a still allows a topological order

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 2 and q == [\"a\"] and r == 2, \"p, q and r\"\n    assert s == 4 and t is False, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a small build tool

Build tools (like `make`) keep a list of rules, "this target is built from these inputs", and rebuild only what's affected when something changes. Write:

- `parse_rules(text)`: lines `target: input input ...` (an input may be another target or a source file; blank lines and `#` comments ignored); return a dict from target to its list of inputs. Raise `ValueError` mentioning the line number for a line without `:` or with an empty target, or a target defined twice.
- `build_order(rules)`: every target in an order where each comes after any target among its inputs; among targets ready at the same time, alphabetical (a heap). Raise `ValueError` naming the targets in a cycle (sorted), found as those never ready.
- `affected(rules, changed)`: given a set of changed **files** (sources or targets), the targets that need rebuilding, in build order: every target with a changed input, and, in turn, everything built from those.

The checks use a 2,000-target chain.

In [ ]:
import heapq

def parse_rules(text):
    ___

def build_order(rules):
    ___

def affected(rules, changed):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("RULES = \"\"\"# a small website\nsite: index.html about.html style.css\nindex.html: index.md header.html\nabout.html: about.md header.html\nheader.html: header.md\nstyle.css: style.scss\n\"\"\"\ndef test_parse():\n    \"rules\"\n    r = parse_rules(RULES)\n    assert r[\"site\"] == [\"index.html\", \"about.html\", \"style.css\"] and r[\"header.html\"] == [\"header.md\"] and len(r) == 5\n    for bad in [\"no colon here\", \": x\", \"a: b\\na: c\"]:\n        try:\n            parse_rules(bad)\n            assert False, bad\n        except ValueError as e:\n            assert \"line\" in str(e)\ndef test_order():\n    \"build order\"\n    r = parse_rules(RULES)\n    assert build_order(r) == [\"header.html\", \"about.html\", \"index.html\", \"style.css\", \"site\"]\n    try:\n        build_order({\"a\": [\"b\"], \"b\": [\"c\"], \"c\": [\"a\"], \"d\": [\"a\"]})\n        assert False\n    except ValueError as e:\n        assert \"a\" in str(e) and \"b\" in str(e) and \"c\" in str(e)\n    assert affected(r, {\"header.md\"}) == [\"header.html\", \"about.html\", \"index.html\", \"site\"]\n    assert affected(r, {\"style.scss\"}) == [\"style.css\", \"site\"] and affected(r, {\"nothing.txt\"}) == []\n    assert affected(r, {\"index.html\"}) == [\"site\"]\ndef test_chain():\n    \"long chains\"\n    rules = {f\"t{i}\": [f\"t{i + 1}\"] for i in range(2000)}\n    rules[\"t2000\"] = [\"src\"]\n    order = build_order(rules)\n    assert order[0] == \"t2000\" and order[-1] == \"t0\"\n    assert len(affected(rules, {\"src\"})) == 2001")

## Warm-up

That's a topological sort by brute force: keep picking something whose requirements are all done. It's O(V²) or worse, re-checking everything each time. Today's algorithms do it in O(V + E).

## Orders that respect dependencies

Draw an edge u → v for "u must come before v". A **topological order** lists the vertices so that every edge points **forwards**: u appears before v for every edge u → v.

- If the graph has a **cycle**, there's no topological order: in a cycle A → B → C → A, A must come before itself.
- If the graph is a **DAG**, there always is one: some vertex has no incoming edges (otherwise, walking backwards along incoming edges forever would eventually repeat a vertex, a cycle), so it can go first; remove it, and repeat.

There are usually **many** valid orders: socks and pants can go in either order. Any order respecting the edges is correct, so if a problem needs one particular order (alphabetical among the free choices, say), it must say so.

Be careful about edge direction: "shoes need socks" is the edge **socks → shoes** (socks before shoes). Getting this backwards produces the reversed order.

## Kahn's algorithm

Arthur Kahn's algorithm (1962) is the warm-up made efficient:

1. Count each vertex's **in-degree** (incoming edges: things it's still waiting for).
2. Put every vertex with in-degree 0 in a queue: they're ready now.
3. Repeatedly take a ready vertex, add it to the order, and for each edge out of it, lower that neighbour's in-degree by 1; a neighbour reaching 0 becomes ready.
4. If the order ends up with fewer than V vertices, the rest are stuck waiting on each other: the graph has a **cycle**.

Each vertex is queued once and each edge lowers one count once: **O(V + E)**. To get the alphabetically smallest order, or any order with a rule for choosing among ready vertices, use a **heap** instead of a queue: O(V log V + E).

## From depth-first search

Wednesday's DFS gives another method. When DFS **finishes** a vertex, everything reachable from it has already finished. So in a DAG, every edge u → v has v finishing before u. Reverse the finish order, and every edge points forwards: a topological order. (If DFS meets a grey vertex, there's a cycle and no order.) The method appears in Tarjan's 1976 work on DFS; it costs O(V + E) too.

Kahn's algorithm is easier to explain and naturally reports cycles and "what's ready now"; the DFS version is shorter when you already have a DFS, and it's the first pass of Thursday's strongly connected components.

## graphlib

Since Python 3.9 the standard library has `graphlib.TopologicalSorter`. You give it, for each node, the nodes it **depends on** (its predecessors), the "needs" direction. `static_order()` returns a valid order, and a cycle raises `graphlib.CycleError` with the cycle in its arguments. It can also hand out work in rounds for running tasks in parallel: `prepare()`, then repeatedly `get_ready()` (everything currently free) and `done(...)` as tasks complete.

The rounds are the same idea as Kahn's algorithm, one layer at a time: everything in a round can happen at once.

## Working along the order

Once vertices are in topological order, many questions about a DAG become one pass, because when you reach a vertex, everything before it has already been dealt with. This is a first taste of **dynamic programming**, a later week's topic.

- **Levels**: the fewest rounds (semesters, build steps) needed if everything ready can happen at once: level(v) = 1 + the largest level among the things v needs.
- **Longest path**: in a DAG (and only there, since cycles would make paths endless), the longest path is found the same way. In general graphs, longest simple paths are among the hardest problems known.
- **Critical path**: give each task a duration. A task's earliest finish is its duration plus the latest earliest-finish among the tasks it needs. The project's length is the largest earliest finish, and the tasks along the chain that sets it are the **critical path**: delay any of them and the whole project is late. The **critical path method** was developed in the late 1950s by Morgan Walker and James Kelley for planning large building projects.

## Practice

## Quiz

## Daily challenge

## Before you go

Write down five things you do every morning and which must come before which, then find two different valid orders. Tomorrow is the Forge: a transport network planner that uses this whole week.